# Scalar Sum (warmup)

Bot task id: `00_scalar_sum_two_numbers`, deadline 06.12.2026 00:00 MSK. [Open in Colab](https://colab.research.google.com/github/PE51K/itmo-ai-talent-hub-efficient-models-course/blob/main/hw3/00_scalar_sum_two_numbers.ipynb)

## Условие

Нужно сложить два скалярных тензора `x` и `y` на GPU и вернуть результат $z = x + y$.

**Требования**
- одно простейшее Triton-ядро;
- загрузить по указателям `x_ptr`, `y_ptr`, записать сумму в `z_ptr`.

В сигнатуре не хватает одной строки.

## Setup

Runtime -> Change runtime type -> T4 GPU.

In [ ]:
import importlib.util

import torch
import triton

assert torch.cuda.is_available(), "no GPU in this runtime: Runtime -> Change runtime type -> T4 GPU"
print(f"torch {torch.__version__}, triton {triton.__version__}, {torch.cuda.get_device_name()}")

## Solution

`%%writefile` saves the cell to `00_scalar_sum_two_numbers.py`, the file to submit. Run it after every edit, then the check.

In [ ]:
%%writefile 00_scalar_sum_two_numbers.py
import torch
import triton
import triton.language as tl


@triton.jit
def _sum2_kernel(x_ptr, y_ptr, z_ptr):
    x = tl.load(x_ptr)
    y = tl.load(y_ptr)
    # TODO: единственная строка решения
    tl.store(z_ptr, z)


def sum2_triton(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    assert x.is_cuda and y.is_cuda, 'Оба скаляра должны быть на CUDA'
    assert x.numel() == 1 and y.numel() == 1, 'Ожидаются скаляры'
    z = torch.empty_like(x)
    _sum2_kernel[(1,)](x, y, z)
    return z

## Check

Compares with PyTorch on shapes with tails and checks some of the requirements. These are my tests, not the bot's: passing them doesn't guarantee the bot accepts the solution.

In [ ]:
PATH = "00_scalar_sum_two_numbers.py"
spec = importlib.util.spec_from_file_location("sol", PATH)
sol = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sol)
SRC = open(PATH).read()

x, y = torch.tensor([3.5], device="cuda"), torch.tensor([-1.25], device="cuda")
torch.testing.assert_close(sol.sum2_triton(x, y), x + y)

torch.cuda.synchronize()
print("PASS")

## Download

In [ ]:
from google.colab import files

files.download("00_scalar_sum_two_numbers.py")